# Análisis de la colección de proyectos de la PAE





Montamos la unidad de google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import pandas as pd

os.chdir("/content/drive/MyDrive/PROYECTOS/PAE")
print("DIRECTORIO DE TRABAJO:", os.getcwd())
#os.listdir()

# Nombre del fichero que contiene los proyectos
PROJECTS_XLSX = "data/EUproject.xlsx"

df_proj = pd.read_excel(PROJECTS_XLSX)

print("Shape:", df_proj.shape)
print("Columnas:", list(df_proj.columns))

df_proj.head()

DIRECTORIO DE TRABAJO: /content/drive/MyDrive/PROYECTOS/PAE
Shape: (79840, 3)
Columnas: ['title', 'description', 'Nivel_1']


,title,description,Nivel_1
0,‘Till Death Us Do Part’: The Comparative Histo...,This project offers new complex analysis of th...,['social sciences']
1,Development and Manufacture Scoop Intake and C...,B.1.1 Concept and Objectives The concept of th...,NaN
2,Contribution of Compact Neighbourhoods to Soci...,This research investigates how compact urban f...,NaN
3,Archives of Early Human Occupation in Western ...,"""A number of important archaeological sites wi...","['social sciences', 'humanities']"
4,Design and Expansion Turbine for Domestic Refr...,This proposal for a Marie Curie Intra-European...,['natural sciences']


In [3]:
df_proj["Nivel_1"].unique()

array(["['social sciences']", nan, "['social sciences', 'humanities']",
       "['natural sciences']", "['engineering and technology']",
       "['engineering and technology', 'medical and health sciences']",
       "['natural sciences', 'humanities']",
       "['social sciences', 'natural sciences']", "['humanities']",
       "['medical and health sciences']", "['agricultural sciences']",
       "['engineering and technology', 'humanities']",
       "['social sciences', 'engineering and technology']",
       "['natural sciences', 'engineering and technology']",
       "['social sciences', 'natural sciences', 'medical and health sciences']",
       "['natural sciences', 'medical and health sciences']",
       "['social sciences', 'medical and health sciences', 'humanities']",
       "['agricultural sciences', 'natural sciences']",
       "['agricultural sciences', 'engineering and technology']",
       "['agricultural sciences', 'natural sciences', 'engineering and technology']",
     

In [4]:
df_filtered = df_proj[
    df_proj["Nivel_1"]
        .astype(str)
        .str.contains("engineering and technology", case=False, na=False)
].copy()

print("Número de proyectos con engineering and technology:", len(df_filtered))
df_filtered.head()




Número de proyectos con engineering and technology: 27833


,title,description,Nivel_1
9,"""Strategic Energy Technology Plan 2011 confere...","""The Strategic Energy Technology Plan is the t...",['engineering and technology']
21,Creating a Central European Thermal Water Rese...,Central Eastern Europe’s rural areas have been...,['engineering and technology']
25,Tilt Rotor ATM Integrated Validation of Enviro...,"""Cleansky-Green Rotor Craft sub-project 5 is a...","['engineering and technology', 'medical and he..."
74,Advanced Methodologies for the Determination o...,Much of Western Europe has inherited soil cont...,['engineering and technology']
83,Development of new nanocomposites using materi...,Silver nanoparticles and silver based nanostr...,['engineering and technology']


In [5]:
import re
import unicodedata
import pandas as pd

# Regex para artefactos típicos de Excel
_x000d_re = re.compile(r"_x000D_?", flags=re.IGNORECASE)
_ws_re = re.compile(r"\s+")

def clean_excel_text(text):
    if pd.isna(text):
        return ""

    s = str(text)

    # 1️⃣ Eliminar artefactos Excel tipo _x000D_ o _x000D
    s = _x000d_re.sub(" ", s)

    # 2️⃣ Eliminar secuencia literal "\n"
    s = s.replace("\\n", " ")

    # 3️⃣ Reemplazar saltos reales y tabs
    s = s.replace("\r", " ").replace("\n", " ").replace("\t", " ")

    # 4️⃣ Normalizar Unicode (quita formas raras)
    s = unicodedata.normalize("NFKC", s)

    # 5️⃣ Eliminar caracteres de control invisibles (Unicode categoría C)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")

    # 6️⃣ Reemplazar NBSP
    s = s.replace("\u00A0", " ")

    # 7️⃣ Normalizar espacios múltiples
    s = _ws_re.sub(" ", s).strip()

    return s


In [6]:
df_filtered["title"] = df_filtered["title"].apply(clean_excel_text)
df_filtered["description"] = df_filtered["description"].apply(clean_excel_text)

# Crear campo combinado listo para embeddings
df_filtered["text"] = (
    df_filtered["title"] + ". " + df_filtered["description"]
).str.strip()

df_filtered.head()

,title,description,Nivel_1,text
9,"""Strategic Energy Technology Plan 2011 confere...","""The Strategic Energy Technology Plan is the t...",['engineering and technology'],"""Strategic Energy Technology Plan 2011 confere..."
21,Creating a Central European Thermal Water Rese...,Central Eastern Europe’s rural areas have been...,['engineering and technology'],Creating a Central European Thermal Water Rese...
25,Tilt Rotor ATM Integrated Validation of Enviro...,"""Cleansky-Green Rotor Craft sub-project 5 is a...","['engineering and technology', 'medical and he...",Tilt Rotor ATM Integrated Validation of Enviro...
74,Advanced Methodologies for the Determination o...,Much of Western Europe has inherited soil cont...,['engineering and technology'],Advanced Methodologies for the Determination o...
83,Development of new nanocomposites using materi...,Silver nanoparticles and silver based nanostru...,['engineering and technology'],Development of new nanocomposites using materi...


In [7]:
df_filtered[df_filtered["text"].str.contains("_x000D", case=False, na=False)]


,title,description,Nivel_1,text


In [8]:
from google.colab import files

OUTPUT_FILE = "data/EUproject_engineering_filtered.csv"

df_filtered.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"   # importante para que Excel lo abra bien
)

print(f"Fichero generado: {OUTPUT_FILE}")
print(f"Número de filas: {len(df_filtered)}")

#files.download(OUTPUT_FILE)


Fichero generado: data/EUproject_engineering_filtered.csv
Número de filas: 27833


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
df_filtered.shape

(27833, 4)